# 1교시 | OT 및 개발환경 구축
**환경**: Google Colab | **방법**: 이론 + 실습

---

## 학습 목표

- 과정의 전체 구조와 2일간의 학습 흐름을 이해한다.
- Google Colab에서 실습 환경을 구성하고 정상 작동을 확인한다.
- OpenAI API Key를 안전하게 등록하고 호출 테스트를 완료한다.

### 처음 읽는 분을 위한 안내

- 이번 교시를 한 줄로 보면: 앞으로 2일간의 학습 흐름과 실습 환경을 미리 안정화하는 시간이다.
- 처음에는 여기까지 이해하면 충분하다: 왜 Colab과 RunPod를 나눠 쓰는지, 왜 GPU(Graphics Processing Unit)와 API Key(Application Programming Interface Key) 확인이 중요한지만 잡아도 된다.
- 헷갈려도 괜찮은 부분: 아직 모델 구조를 깊게 모르더라도 괜찮다. 이번 교시는 "환경이 정상인지 확인하는 법"을 익히는 것이 우선이다.

### 용어 미니사전

| 용어 | 아주 쉽게 말하면 |
|---|---|
| Colab | 브라우저에서 바로 쓰는 실습용 파이썬 환경 |
| RunPod | 더 큰 GPU 자원을 빌려 쓰는 클라우드 실습 환경 |
| GPU(Graphics Processing Unit) | 모델 연산을 빠르게 처리해 주는 장치 |
| API Key(Application Programming Interface Key) | 외부 모델 서비스를 안전하게 호출하기 위한 비밀 키 |
| Secrets | 키를 코드에 직접 쓰지 않고 따로 저장하는 방법 |
| VRAM(Video RAM) | GPU(Graphics Processing Unit)가 모델과 데이터를 올려두는 메모리 |

---

## 이론

### 먼저 큰 그림부터 이해하기

이 과정의 목표는 LLM/VLM을 단순히 "잘 쓰는 방법"으로 배우는 것이 아니라, **왜 그런 출력이 나오고 내부에서 어떤 구조가 작동하는지 이해하는 것**이다.  
즉, 이 1교시는 단순 오리엔테이션이 아니라 이후 2일간의 모든 실습이 흔들리지 않도록 **학습 지도와 실행 환경을 한 번에 맞추는 출발점**이라고 보면 된다.

이 문서를 읽을 때는 아래 세 가지를 먼저 잡아 두면 좋다.

1. 앞으로 2일간 무엇을 어떤 순서로 배우는가?
2. 왜 Colab과 RunPod를 나눠서 쓰는가?
3. 실습 환경이 왜 이론 이해만큼 중요한가?

### 과정 전체 구조

이 과정은 LLM/VLM을 "블랙박스"가 아닌 구조적으로 이해하는 것을 목표로 한다. 단순히 API를 호출하는 수준을 넘어, 모델 내부에서 어떤 연산이 일어나는지를 실험으로 직접 관찰한다.

**2일간 학습 흐름**

```
[1일차] 텍스트 기반 구조 이해
  Transformer → Attention → LLM 생성 원리 → 추론 전략 → KV Cache → ViT

[2일차] 멀티모달 확장 및 최신 모델 실습
  CLIP → LLM 추론 + Captioning → VQA → RunPod 전환 → Qwen3-VL 구조 분석
```

이 흐름은 임의로 나열된 것이 아니다.  
1일차에는 먼저 텍스트 기반 Transformer와 추론 구조를 이해하고, 그 다음 이미지 토큰화와 CLIP으로 넘어간다. 2일차에는 이 지식을 바탕으로 멀티모달 입력과 최신 VLM 구조를 다룬다. 즉, **텍스트 구조 이해 → 비전 이해 → 멀티모달 통합** 순서로 난이도가 쌓이도록 설계되어 있다.

### 환경 구성 전략

| 환경 | 사용 교시 | 특징 |
|---|---|---|
| **Google Colab (무료)** | 1~10교시, 13교시 | T4 GPU(15GB), 브라우저 접속, 빠른 시작 |
| **RunPod A40** | 11~12교시 | 48GB VRAM, JupyterLab URL 접속, Qwen3-VL 실행 |


환경을 나누는 이유도 분명히 이해할 필요가 있다.  
Colab은 빠르게 시작하고 대부분의 기초 실습을 진행하기에 충분하지만, Qwen3-VL 같은 대형 멀티모달 모델은 더 큰 VRAM과 안정적인 환경이 필요하다. 그래서 초반 학습은 Colab에서, 후반 대형 모델 실습은 RunPod A40에서 진행한다.

### Google Colab 사용 시 주의사항

- **런타임 유형**: 반드시 `T4 GPU`로 설정 (런타임 → 런타임 유형 변경)
- **세션 유지**: 12시간 후 자동 종료, 변수/설치 패키지 초기화됨
- **API Key 보안**: 코드에 직접 입력하지 않고 **Colab Secrets** 사용
- **패키지 설치**: 세션마다 재설치 필요 (`!pip install`)

즉, 이 교시의 환경 설정은 단순 준비 작업이 아니라, 이후 실습에서 생길 수 있는 오류를 미리 줄이기 위한 사전 안정화 단계라고 보면 된다.

---

## 실습

### 자주 막히는 오류

- `torch.cuda.is_available()`가 `False`: Colab 런타임이 GPU가 아니라 CPU로 잡힌 경우가 많다.
- API Key 로드 오류: Secrets 이름이 정확히 `OPENAI_API_KEY`인지 먼저 확인한다.
- 패키지 import 실패: Colab 세션이 재시작되면 설치가 사라질 수 있으므로 설치 셀을 다시 실행한다.
- RunPod 접속이 안 됨: 계정 로그인과 Pod 상태를 먼저 확인하고, JupyterLab 버튼이 맞는지 본다.

### 실패했을 때 체크 순서

1. 런타임과 준비 상태부터 확인한다: GPU 연결, 패키지 설치, API Key 등록 여부를 먼저 본다.
2. 직전 셀 출력에서 첫 번째 에러 문장을 읽는다: `ModuleNotFoundError`, `KeyError`, `False` 같은 핵심 신호만 먼저 잡는다.
3. 문제 난 단계만 다시 실행한다: 그래도 안 되면 런타임을 다시 연결하고 Step 1부터 순서대로 재확인한다.

실습은 아래 순서대로 **한 단계씩 실행**하는 것을 권장한다.  
이번 교시의 목표는 크게 세 가지다.

1. Colab GPU 환경이 정상인지 확인하기
2. OpenAI API Key를 안전하게 등록하고 호출 테스트하기
3. 2일차 RunPod 전환 전에 필요한 준비를 점검하기

각 단계는 이후 모든 실습의 기반이 되므로, 한 단계씩 실행하고 출력 결과를 꼭 확인하는 편이 좋다.

### Step 1. Colab 런타임 설정 확인

```
상단 메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU 선택 → 저장
```

이 단계에서 확인할 점:

- 런타임이 CPU가 아니라 T4 GPU로 설정되었는가?
- 이후 실습이 모두 이 GPU 환경을 기준으로 진행된다는 점을 이해했는가?

### Step 2. OpenAI API Key 등록 (Colab Secrets)

```
왼쪽 사이드바 🔑 아이콘 → + 새 보안 비밀 추가
  이름: OPENAI_API_KEY
  값: sk-...
```

이 단계에서 확인할 점:

- API Key를 코드에 직접 쓰지 않고 Secrets에 저장했는가?
- 환경 변수로 불러오는 방식이 더 안전하다는 점을 이해하는가?

### Step 3. 환경 설정 노트북

In [ ]:
# ── 패키지 설치 ────────────────────────────────────────────────
!pip install -q \
    transformers==4.41.0 \
    torch torchvision \
    openai \
    pillow matplotlib seaborn \
    requests ipywidgets

# ── API Key 로드 ───────────────────────────────────────────────
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# ── 환경 확인 ──────────────────────────────────────────────────
import torch
import transformers
import openai

print("=" * 40)
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"openai        : {openai.__version__}")
print(f"CUDA 사용 가능 : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM          : {total:.1f} GB")
print("=" * 40)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 130.0 MB/s eta 0:00:00
torch         : 2.10.0+cu128
transformers  : 4.41.0
openai        : 2.32.0
CUDA 사용 가능 : True
GPU           : Tesla T4
VRAM          : 14.6 GB


이 단계에서 확인할 점:

- `torch.cuda.is_available()`가 `True`인지 확인
- GPU 이름이 Tesla T4로 보이는지 확인
- 이후 실습에 필요한 주요 패키지가 정상 import되는지 확인

### Step 4. OpenAI API 호출 테스트

In [ ]:
from openai import OpenAI

client = OpenAI()  # OPENAI_API_KEY 환경변수 자동 참조

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "한 문장으로 LLM을 설명해줘."}],
    max_tokens=100,
)

print(response.choices[0].message.content)
print(f"\n사용 토큰: {response.usage.total_tokens}")

LLM(대형 언어 모델)은 방대한 텍스트 데이터를 학습하여 사람처럼 자연스럽게 언어를 이해하고 생성할 수 있는 인공지능 모델입니다.

사용 토큰: 56


이 단계에서 확인할 점:

- API 호출이 실제로 성공하는가?
- 응답 텍스트와 토큰 사용량이 정상적으로 출력되는가?
- 이후 실습에서 API 오류가 나면 환경 문제인지 모델 문제인지 구분할 수 있겠는가?

### 실습을 마친 뒤 정리

위 실습이 끝나면 아래 내용을 말로 설명할 수 있어야 한다.

1. 왜 1일차는 Colab, 2일차 일부는 RunPod를 쓰는가?
2. 왜 API Key를 Secrets로 관리해야 하는가?
3. 왜 첫 교시에서 GPU와 패키지 확인을 반드시 해야 하는가?
4. 환경 점검이 이후 이론 실습의 실패를 줄이는 이유는 무엇인가?

---

## 체크리스트

- [ ] Colab T4 GPU 런타임 연결 확인
- [ ] `torch.cuda.is_available()` → `True`
- [ ] OpenAI API Key Secrets 등록 및 호출 성공

### 혼자 점검하기

아래 질문에 막힘 없이 답하면 1교시 준비가 제대로 끝난 것이다.

1. 이 과정이 `API 사용법`이 아니라 `구조 이해` 중심이라는 말은 무슨 뜻인가?
2. Colab과 RunPod를 왜 나눠서 사용하는가?
3. Colab Secrets를 사용하는 이유는 무엇인가?
4. `torch.cuda.is_available()` 확인이 왜 중요한가?
5. 첫 교시 환경 점검이 이후 수업 품질에 어떤 영향을 주는가?

---

## 다음 교시 예고

**2교시**: GPT-2를 직접 분해하며 Transformer의 구성 요소(레이어, 헤드, 임베딩 차원)를 확인한다.

---

## 부록 | 관찰 과제·혼자 점검 모범 답안

이 부록은 교육생이 1교시를 단순 환경 세팅 시간으로 넘기지 않고, **왜 이 준비가 전체 과정의 기반인지**를 스스로 설명할 수 있도록 돕는 해설이다.

### 관찰 과제 모범 답안

**1. GPU가 실제로 잡히는지 확인**

`torch.cuda.is_available()`가 `True`이고 GPU 이름이 Tesla T4로 보인다면, 이후 PyTorch 기반 실습을 GPU에서 수행할 준비가 된 것이다. 이 확인이 중요한 이유는, GPU가 잡히지 않으면 속도 저하나 라이브러리 동작 문제를 나중에 모델 문제로 오해할 수 있기 때문이다.

**2. API 호출이 실제로 성공하는지 확인**

단순히 Key를 등록했다고 끝이 아니라, 실제 호출이 성공해야 환경이 완성된 것이다. 응답 텍스트와 토큰 사용량이 출력되면 인증과 네트워크, SDK 사용이 모두 정상이라는 뜻이다.

### 혼자 점검하기 모범 답안

**1. 이 과정이 `API 사용법`이 아니라 `구조 이해` 중심이라는 말은 무슨 뜻인가?**

단순히 모델을 호출해 결과를 받는 데서 끝나지 않고, 내부에서 어떤 구조와 연산이 작동하는지까지 이해하는 것을 목표로 한다는 뜻이다. 즉, 출력만 보는 것이 아니라 왜 그런 출력이 나오는지 설명할 수 있어야 한다.

**2. Colab과 RunPod를 왜 나눠서 사용하는가?**

Colab은 빠르게 시작하고 기초 실습을 진행하기에 편리하다. 반면 RunPod A40은 더 큰 VRAM과 안정적인 환경을 제공해 Qwen3-VL 같은 대형 모델 실습에 적합하다. 따라서 학습 난이도와 자원 요구에 맞춰 환경을 나눠 쓰는 것이다.

**3. Colab Secrets를 사용하는 이유는 무엇인가?**

API Key를 코드에 직접 적으면 노출 위험이 커지고, 노트북 공유 시 보안 문제가 생길 수 있다. Secrets를 사용하면 키를 코드 밖에서 안전하게 관리할 수 있다.

**4. `torch.cuda.is_available()` 확인이 왜 중요한가?**

이 값이 `True`여야 GPU 가속을 사용할 수 있기 때문이다. 만약 `False`라면 런타임 설정이나 드라이버, 환경 구성이 잘못되었을 가능성이 있어 먼저 바로잡아야 한다.

**5. 첫 교시 환경 점검이 이후 수업 품질에 어떤 영향을 주는가?**

환경이 안정적이면 이후 실습에서 오류 원인을 더 쉽게 분리할 수 있고, 수업 시간을 모델 이해와 실험에 집중할 수 있다. 반대로 환경 점검이 부족하면 단순 설정 문제로 전체 흐름이 자주 끊길 수 있다.

### 활용 팁

1. 첫 교시에서는 코드를 많이 치기보다 환경이 정상인지 확인하는 데 집중한다.
2. 오류가 생기면 모델 문제보다 먼저 GPU, 패키지, API Key 상태를 본다.
3. 이후 교육생 지도 시에도 환경 점검 체크리스트를 먼저 주는 것이 효율적이다.